# Video Model Traning

## Module Imports

In [1]:
!pip install -q transformers datasets torchaudio accelerate evaluate soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00a 0:00:01


In [2]:
import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import (
    AutoFeatureExtractor,
    AutoModelForAudioClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
 
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"   # ~95M params, fast, good accuracy for spoof detection
DATASET_NAME = "ASVspoof/ASVspoof2019_LA"  # standard real-vs-spoof audio benchmark
SAMPLE_RATE = 16000                     # wav2vec2 expects 16kHz audio
MAX_DURATION_SECONDS = 4                # clip/pad every sample to 4s -> keeps training fast on free GPU
OUTPUT_DIR = "./audio_deepfake_model"

In [ ]:
# The dataset has a "key" or "label" column: bonafide (real) vs spoof (fake).
# We map it to 0 = real, 1 = fake for consistency with the video model.
dataset = load_dataset(DATASET_NAME)
 
# Resample audio column to 16kHz (required by wav2vec2)
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
 
LABEL2ID = {"real": 0, "fake": 1}
ID2LABEL = {0: "real", 1: "fake"}
 
def normalize_label(example):
    # ASVspoof uses "bonafide" for real and "spoof" for fake — adjust key name if your
    # loaded version differs (check dataset["train"].features to confirm column names).
    raw_label = example["label"] if "label" in example else example["key"]
    is_fake = str(raw_label).lower() != "bonafide"
    example["labels"] = LABEL2ID["fake"] if is_fake else LABEL2ID["real"]
    return example
 
dataset = dataset.map(normalize_label)

In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
 
def preprocess(batch):
    audio_arrays = [x["array"] for x in batch["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        max_length=int(SAMPLE_RATE * MAX_DURATION_SECONDS),
        truncation=True,
        padding="max_length",
    )
    return inputs
 
encoded_dataset = dataset.map(
    preprocess,
    remove_columns=["audio"],
    batched=True,
    batch_size=8,
)

In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")
 
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels)["f1"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,     # keep small for free-tier VRAM
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,     # effective batch size = 16
    num_train_epochs=5,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                         # mixed precision -> ~2x faster, fits in free GPU VRAM
    report_to="none",
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"] if "validation" in encoded_dataset else encoded_dataset["test"],
    compute_metrics=compute_metrics,
)
 
trainer.train()
 

In [ ]:
rainer.save_model(OUTPUT_DIR)
feature_extractor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [ ]:
# # Optional: push to your own Hugging Face Hub repo so you can load it later
# from huggingface_hub import login
# login(HF_TOKEN)  # paste your HF token
# trainer.push_to_hub("SatyamVish123/audio-deepfake-detector")